In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


df = pd.read_csv("lab_ticket_resolution.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

print("Data types:")
print(df.dtypes)

print("Descriptive statistics:")
display(df.describe(include="all"))

print("Missing values:")
print(df.isna().sum())

In [ ]:
X = df[["complexity_score", "team_experience_years"]]

y = df["resolution_hours"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("Features:")
display(X.head())

print("Target:")
display(y.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


In [ ]:
baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_predictions)
)
baseline_r2 = r2_score(y_test, baseline_predictions)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

In [ ]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = np.sqrt(
    mean_squared_error(y_test, linear_predictions)
)
linear_r2 = r2_score(y_test, linear_predictions)

print("Linear Regression MAE:", linear_mae)
print("Linear Regression RMSE:", linear_rmse)
print("Linear Regression R²:", linear_r2)

print("Intercept:", linear_model.intercept_)

print("\nFeature coefficients:")
for feature, coefficient in zip(X.columns, linear_model.coef_):
    print(f"{feature}: {coefficient:.4f}")

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for degree in [2, 3]:
    model = make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        LinearRegression()
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error"
    )

    mae_scores = -scores

    print(f"Degree {degree} mean CV MAE: {mae_scores.mean():.4f} hours")
    print(
        f"Degree {degree} CV MAE standard deviation: "
        f"{mae_scores.std():.4f} hours"
    )
    print()

In [ ]:
ridge_pipeline = make_pipeline(
    StandardScaler(),
    Ridge()
)

lasso_pipeline = make_pipeline(
    StandardScaler(),
    Lasso(max_iter=10000)
)

alpha_values = [0.01, 0.1, 1, 10, 100]


ridge_search = GridSearchCV(
    ridge_pipeline,
    {"ridge__alpha": alpha_values},
    cv=cv,
    scoring="neg_mean_absolute_error"
)

ridge_search.fit(X_train, y_train)

print("Best Ridge alpha:", ridge_search.best_params_["ridge__alpha"])
print("Best Ridge CV MAE:", -ridge_search.best_score_)


lasso_search = GridSearchCV(
    lasso_pipeline,
    {"lasso__alpha": alpha_values},
    cv=cv,
    scoring="neg_mean_absolute_error"
)

lasso_search.fit(X_train, y_train)

print("Best Lasso alpha:", lasso_search.best_params_["lasso__alpha"])
print("Best Lasso CV MAE:", -lasso_search.best_score_)

In [ ]:
ridge_predictions = ridge_search.predict(X_test)
lasso_predictions = lasso_search.predict(X_test)

print("RIDGE")
print("MAE:", mean_absolute_error(y_test, ridge_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, ridge_predictions)))
print("R²:", r2_score(y_test, ridge_predictions))

print("\nLASSO")
print("MAE:", mean_absolute_error(y_test, lasso_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, lasso_predictions)))
print("R²:", r2_score(y_test, lasso_predictions))

In [ ]:
poly2_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    LinearRegression()
)

poly2_model.fit(X_train, y_train)
poly2_predictions = poly2_model.predict(X_test)

poly2_mae = mean_absolute_error(y_test, poly2_predictions)
poly2_rmse = np.sqrt(mean_squared_error(y_test, poly2_predictions))
poly2_r2 = r2_score(y_test, poly2_predictions)

print("Polynomial Degree 2")
print("MAE:", poly2_mae)
print("RMSE:", poly2_rmse)
print("R²:", poly2_r2)


poly3_model = make_pipeline(
    PolynomialFeatures(degree=3, include_bias=False),
    StandardScaler(),
    LinearRegression()
)

poly3_model.fit(X_train, y_train)
poly3_predictions = poly3_model.predict(X_test)

poly3_mae = mean_absolute_error(y_test, poly3_predictions)
poly3_rmse = np.sqrt(mean_squared_error(y_test, poly3_predictions))
poly3_r2 = r2_score(y_test, poly3_predictions)

print("\nPolynomial Degree 3")
print("MAE:", poly3_mae)
print("RMSE:", poly3_rmse)
print("R²:", poly3_r2)

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(y_test, linear_predictions)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--"
)

plt.xlabel("Actual Resolution Hours")
plt.ylabel("Predicted Resolution Hours")
plt.title("Linear Regression: Actual vs Predicted")

plt.show()

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Mean Baseline",
        "Linear Regression",
        "Polynomial Degree 2",
        "Polynomial Degree 3",
        "Ridge",
        "Lasso"
    ],
    "MAE": [
        baseline_mae,
        linear_mae,
        poly2_mae,
        poly3_mae,
        mean_absolute_error(y_test, ridge_predictions),
        mean_absolute_error(y_test, lasso_predictions)
    ],
    "RMSE": [
        baseline_rmse,
        linear_rmse,
        poly2_rmse,
        poly3_rmse,
        np.sqrt(mean_squared_error(y_test, ridge_predictions)),
        np.sqrt(mean_squared_error(y_test, lasso_predictions))
    ],
    "R2": [
        baseline_r2,
        linear_r2,
        poly2_r2,
        poly3_r2,
        r2_score(y_test, ridge_predictions),
        r2_score(y_test, lasso_predictions)
    ]
})

display(results.round(4))